# Step 5 — Network-Loss Optimization and Final Comparison

Step 5 keeps the Step-4 feasible set but replaces the price objective with a linearized marginal AC-loss objective. Because all depots share one electrical bus, their marginal location coefficient is expected to be the same at a given hour; the remaining flexibility is mainly temporal.

### What this cell does — Load inputs and calculate marginal loss factors

At each hour Pandapower perturbs the shared connection by 0.05 MW and measures the change in AC losses. The local sensitivity is copied to all four companies because they share the same bus.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data import load_fleet, load_prices, load_network_limits, load_base_loads, load_bus_mapping
from src.network.model import build_network
from src.network.losses import calculate_marginal_loss_factors
from src.optimization.model import optimize_loss
from src.network.timeseries import run_timeseries_detailed, build_snapshot_network
from src.network.contingency import run_n1, SWITCH_CONFIGS_ALL_CLOSED
from src.reporting.metrics import scenario_metrics
from src.reporting.plots import save_schedule_plot, save_network_plot

fleet = load_fleet()
prices = load_prices()
network_limits = load_network_limits()
base_p, base_q = load_base_loads()
bus_map = load_bus_mapping(require_shared=True)
network_factory = lambda: build_network(halve_existing_loads=True, close_ring_switches=True)

loss_factors = calculate_marginal_loss_factors(network_factory, base_p, base_q, bus_map, delta_p_mw=0.05)
config.NETWORK_DIR.mkdir(parents=True, exist_ok=True)
loss_factors.to_csv(config.LOSS_FACTORS, index=False)
display(loss_factors.groupby("depot").loss_coefficient.describe())


### What this cell does — Solve loss optimization

PuLP minimizes the marginal-loss proxy under exactly the same availability, charger, energy and Step-3 network constraints used for the cost case. The proxy is not reported as actual network loss; Pandapower supplies the physical loss result.

In [ ]:
schedule, optimization_summary = optimize_loss(fleet, network_limits, loss_factors)
display(optimization_summary)


### What this cell does — Detailed AC validation and line-loss breakdown

Runs the optimized schedule through full AC power flow and also records losses per line for every hour. This satisfies the Step-5 requirement to identify where losses concentrate.

In [ ]:
results, line_losses = run_timeseries_detailed(network_factory, base_p, base_q, schedule, bus_map)
line_loss_summary = line_losses.groupby("line", as_index=False).agg(loss_MWh=("loss_MW", "sum"), max_loading_percent=("loading_percent", "max")).sort_values("loss_MWh", ascending=False)
display(line_loss_summary.head(10))


### What this cell does — N-1 and worst-contingency losses

Evaluates all-closed N-1 at the peak loss-optimized charging hour. `total_losses_MW` in the contingency table makes it possible to identify the worst contingency loss case without permanently changing the network.

In [ ]:
peak_time = results.loc[results.total_ev_charging_MW.idxmax(), "time"]
peak_net = build_snapshot_network(network_factory, base_p, base_q, schedule, bus_map, peak_time)
n1_summary, n1_violations = run_n1(peak_net, switch_configs=SWITCH_CONFIGS_ALL_CLOSED)
metrics = scenario_metrics("loss_optimized", schedule, prices, results, n1_summary)
worst_contingency = n1_summary.loc[n1_summary.total_losses_MW.idxmax()] if n1_summary.total_losses_MW.notna().any() else None
display(metrics)
if worst_contingency is not None:
    display(worst_contingency.to_frame("value"))


### What this cell does — Final comparison

Combines the three common metric tables. Interpret each strategy against its intended objective: uncontrolled is the operational baseline, cost optimization primarily reduces electricity expenditure, and loss optimization primarily targets AC network efficiency.

In [ ]:
frames = []
for path in [
    config.RESULTS / "step3_uncontrolled" / "scenario_metrics.csv",
    config.RESULTS / "step4_cost_optimization" / "scenario_metrics.csv",
]:
    if path.exists(): frames.append(pd.read_csv(path))
frames.append(metrics)
comparison = pd.concat(frames, ignore_index=True, sort=False)
display(comparison)
summary_dir = config.RESULTS / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(summary_dir / "scenario_comparison.csv", index=False)


### What this cell does — Export Step 5

Writes the loss-optimal schedule, physical AC results, per-line losses, marginal factors, N-1 evidence and plots.

In [ ]:
out = config.RESULTS / "step5_loss_optimization"
out.mkdir(parents=True, exist_ok=True)
schedule.to_csv(out / "loss_optimized_schedule.csv", index=False)
optimization_summary.to_csv(out / "optimization_summary.csv", index=False)
loss_factors.to_csv(out / "marginal_loss_factors.csv", index=False)
results.to_csv(out / "network_results.csv", index=False)
line_losses.to_csv(out / "line_loss_timeseries.csv", index=False)
line_loss_summary.to_csv(out / "line_loss_summary.csv", index=False)
metrics.to_csv(out / "scenario_metrics.csv", index=False)
n1_summary.to_csv(out / "n1_summary.csv", index=False)
n1_violations.to_csv(out / "n1_violations.csv", index=False)
save_schedule_plot(schedule, out / "loss_schedule.png", "Step 5 — Loss-optimal charging")
save_network_plot(results, out / "network_loading.png", "Step 5 — Network loading")
print("Saved to", out)
